In [ ]:
# @title Setup Environment.
from pathlib import Path
import os

lang, branch = 'en', 'main'
scripts_dir = Path.home() / 'ANXETY' / 'scripts'
output = scripts_dir / 'setup.py'

os.makedirs(output.parent, exist_ok=True)
!curl -sLo {output} https://raw.githubusercontent.com/hermeslunanul-ARG/sdLunanul/{branch}/scripts/setup.py

%run $output --lang=$lang --branch=$branch

---
## 1. Widgets 🔽

In [ ]:
# Model selection, vae, control-net and more.
%run $scripts_dir/$lang/widgets-{lang}.py

## 2. Downloading 🔽

In [ ]:
# Downloading libraries, repos, models and more.
%run $scripts_dir/$lang/downloading-{lang}.py

## 3. Start 🔽

In [ ]:
# Launch WebUI | (use -t [m|e|d] for tagger: merged/e621/danbooru)
%run $scripts_dir/launch.py -t d

In [ ]:
# @title 🔍 Diagnóstico Ngrok (paso a paso)
import urllib.request as _req, json, time, os, subprocess as _sp
print('='*60)
print('  🔍 DIAGNÓSTICO NGORK')
print('='*60)

# 1. ¿ngrok está instalado?
_ng = os.popen('which ngrok 2>/dev/null || echo NOT_FOUND').read().strip()
if _ng == 'NOT_FOUND':
    print('  ❌ ngrok NO INSTALADO (no está en PATH)')
    print('  → Necesita reiniciar runtime para reinstalar dependencias')
else:
    print(f'  ✅ ngrok binario: {_ng}')
    print(f'  Versión: {os.popen("ngrok version 2>/dev/null").read().strip() or "N/A"}')

# 2. ¿El token está configurado?
_cfg = os.popen('cat /root/.config/ngrok/ngrok.yml 2>/dev/null || echo NO_FILE').read().strip()
if 'authtoken' in _cfg:
    print('  ✅ Token configurado en ngrok.yml')
else:
    print('  ⚠️ Token NO configurado en ngrok.yml')
    # Check settings.json
    _st = json.load(open('/root/ANXETY/settings.json'))
    _tk = _st.get('WIDGETS', {}).get('ngrok_token', '')
    if _tk:
        print(f'  ✅ Token presente en settings.json (longitud: {len(_tk)})')
    else:
        print('  ❌ Token VACÍO en settings.json — no se guardó en los widgets')

# 3. ¿El puerto 4040 está escuchando?
_p4040 = os.popen('ss -tlnp 2>/dev/null | grep 4040 || netstat -tlnp 2>/dev/null | grep 4040 || echo NOT_LISTENING').read().strip()
if '4040' in _p4040:
    print('  ✅ Puerto 4040 (API ngrok) está escuchando')
    # Try API
    for _ in range(3):
        try:
            _r = _req.urlopen('http://127.0.0.1:4040/api/tunnels', timeout=3)
            _d = json.loads(_r.read())
            if _d.get('tunnels'):
                for _t in _d['tunnels']:
                    print(f'  ✅ Tunnel activo: {_t.get("public_url","N/A")}')
            else:
                print('  ⚠️ API responde pero no hay túneles activos')
            break
        except Exception as _e:
            print(f'  ⏳ API no responde: {_e}')
            time.sleep(2)
else:
    print('  ❌ Puerto 4040 NO está escuchando — ngrok no está corriendo')

print('='*60)


---
# Utilities

In [ ]:
# Run AutoCleaner.
%run $scripts_dir/auto-cleaner.py